In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

# Delete old folder completely
old_path = "/content/drive/MyDrive/code-debugger"
if os.path.exists(old_path):
    shutil.rmtree(old_path)
    print("Old folder deleted")
else:
    print("Nothing to delete")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Create all folders with placeholder files
structure = {
    "/content/drive/MyDrive/code-debugger/ml_service/model": "model folder",
    "/content/drive/MyDrive/code-debugger/backend": "backend folder",
    "/content/drive/MyDrive/code-debugger/frontend": "frontend folder",
    "/content/drive/MyDrive/code-debugger/notebooks": "notebooks folder",
}

for path, desc in structure.items():
    os.makedirs(path, exist_ok=True)
    # Write a real README so Drive registers the folder
    with open(os.path.join(path, "README.txt"), "w") as f:
        f.write(f"This is the {desc} for the code-debugger project.\n")
    print(f" {path}")

print("\nVerifying...")
for path in structure:
    readme = os.path.join(path, "README.txt")
    exists = os.path.exists(readme)
    print(f"  { "done" if exists else 'issue'} {readme}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 /content/drive/MyDrive/code-debugger/ml_service/model
 /content/drive/MyDrive/code-debugger/backend
 /content/drive/MyDrive/code-debugger/frontend
 /content/drive/MyDrive/code-debugger/notebooks

Verifying...
  done /content/drive/MyDrive/code-debugger/ml_service/model/README.txt
  done /content/drive/MyDrive/code-debugger/backend/README.txt
  done /content/drive/MyDrive/code-debugger/frontend/README.txt
  done /content/drive/MyDrive/code-debugger/notebooks/README.txt


In [ ]:
!pip install -q \
    peft==0.10.0 \
    sentence-transformers==2.7.0 \
    datasets==2.19.0 \
    accelerate==0.29.3 \
    gradio

print("Dependencies installed")

Dependencies installed


In [ ]:
import torch
import transformers
import peft
import sentence_transformers
import datasets
import gradio
import groq

# print(f"torch {torch.__version__}")
# print(f"transformers {transformers.__version__}")
# print(f" peft {peft.__version__}")
# print(f" sentence_transformers {sentence_transformers.__version__}")
# print(f" datasets {datasets.__version__}")
# print(f" gradio {gradio.__version__}")
# print(f" groq {groq.__version__}")
# print(f"\n GPU available: {torch.cuda.is_available()}")
# print(f"   Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")

In [ ]:
!groq --version

/bin/bash: line 1: groq: command not found


In [ ]:
import os
from google.colab import userdata

try :
  os.environ["GROQ_API_KEY"]=userdata.get("GROQ_API_KEY")
  print("Groq API key loaded from Colab Secrets")
except:
  os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


Groq API key loaded from Colab Secrets


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

drive_path="/content/drive/MyDrive/code-debugger"
os.makedirs(drive_path, exist_ok=True)
print(f"Google Drive mounted")
print(f"Project folder: {drive_path}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted
Project folder: /content/drive/MyDrive/code-debugger


In [ ]:
import shutil, os

path = "/content/drive/MyDrive/Colab Notebooks/"
files = os.listdir(path)

# Find the exact filename
for f in files:
    if "setup" in f.lower():
        src = os.path.join(path, f)
        dst = "/content/drive/MyDrive/code-debugger/01_setup.ipynb"
        shutil.copy(src, dst)
        print(f"Copied: {repr(f)} → {dst}")

Copied: '01_setup.ipynb' → /content/drive/MyDrive/code-debugger/01_setup.ipynb
Copied: '01_setup.ipynb .ipynb' → /content/drive/MyDrive/code-debugger/01_setup.ipynb


In [ ]:
import os

project = "/content/drive/MyDrive/code-debugger"
for root, dirs, files in os.walk(project):
    level = root.replace(project, "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

code-debugger/
  01_setup.ipynb
  ml_service/
    model/
      README.txt
  backend/
    README.txt
  frontend/
    README.txt
  notebooks/
    README.txt


In [ ]:
!pip install -q groq
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("Ready")

Ready


In [ ]:
import os
import json
import time
from groq import Groq

Client = Groq(api_key=os.environ.get("GROQ_API_KEY"))
LABELS = ["Syntax Error", "Runtime Error", "Logic Bug", "Multiple Issues"]

def generate_synthetic(label_id,label_name,n):
  prompt=f"""Generate {n} different buggy Python code snippets that contain {label_name}s.
  Rules:
- Each snippet must be 2-10 lines long
- Each must be realistic Python code a student might write
- Each must contain exactly the bug type: {label_name}
- Be diverse: use different functions, data structures, algorithms
- Return ONLY a valid JSON array, no explanation, no markdown, like this:
["code snippet 1", "code snippet 2", ...]
  """
  response=Client.chat.completions.create(
      model="llama-3.3-70b-versatile",
      messages=[{"role": "user", "content": prompt}],
      temperature=0.9,
      max_tokens=4096

  )
  raw=response.choices[0].message.content.strip()

  raw = raw.replace("```json", "").replace("```", "").strip()
  try:
    snippets=json.loads(raw)
    return [{"code": s, "label": label_id} for s in snippets if isinstance(s, str)]
  except json.JSONDecodeError:
    print(f" JSON parse failed for {label_name}, got {len(raw)} chars")
    return []


all_generated = []
for i,name in enumerate(LABELS):
  generated=generate_synthetic(i,name,50)
  all_generated.extend(generated)
  print(f"   Got {len(generated)} examples")
  time.sleep(2)
print(f"\nTotal generated: {len(all_generated)}")





















   Got 55 examples
   Got 46 examples
   Got 56 examples
   Got 55 examples

Total generated: 212


In [ ]:
%%writefile /content/drive/MyDrive/code-debugger/ml_service/model/dataset.py
LABELS = ["Syntax Error", "Runtime Error", "Logic Bug", "Multiple Issues"]

RAW_DATA = [
    # SYNTAX ERRORS
    {"code": "def greet(name)\n    print('Hello, ' + name)", "label": 0},
    {"code": "for i in range(10)\n    print(i)", "label": 0},
    {"code": "if x > 0\n    print('pos')\nelse:\n    print('neg')", "label": 0},
    {"code": "class Dog:\n    def __init__(self, name:\n        self.name = name", "label": 0},
    {"code": "def add(a, b):\nreturn a + b", "label": 0},
    {"code": "x = (1 + 2\ny = x * 3", "label": 0},
    {"code": "print('Hello World'", "label": 0},
    {"code": "nums = [1, 2, 3\nprint(nums)", "label": 0},
    {"code": "def divide(a, b):\n   if b != 0:\n  return a / b", "label": 0},
    {"code": "def factorial(n):\n    if n == 0\n        return 1\n    return n * factorial(n-1)", "label": 0},
    {"code": "numbers = [1, 2, 3]\nprint(numbers[10])", "label": 1},
    {"code": "x = 10\ny = 0\nresult = x / y", "label": 1},
    {"code": "name = 'Alice'\nprint(name + 42)", "label": 1},
    {"code": "def greet():\n    print(message)\n\ngreet()", "label": 1},
    {"code": "data = {'name': 'Alice'}\nprint(data['age'])", "label": 1},
    {"code": "text = None\nprint(text.upper())", "label": 1},
    {"code": "import maths\nprint(maths.sqrt(16))", "label": 1},
    {"code": "items = (1, 2, 3)\nitems[0] = 99", "label": 1},
    {"code": "stack = []\nprint(stack.pop())", "label": 1},
    {"code": "def process(data):\n    return data['value'] * 2\n\nprocess(None)", "label": 1},
    {"code": "def is_even(n):\n    return n % 2 == 1", "label": 2},
    {"code": "def is_palindrome(s):\n    return s == s[::-0]", "label": 2},
    {"code": "def celsius_to_fahrenheit(c):\n    return c * 9 / 5 - 32", "label": 2},
    {"code": "def count_vowels(s):\n    count = 0\n    for ch in s:\n        if ch in 'aeiou':\n            count += 1\n    return count - 1", "label": 2},
    {"code": "def max_of_three(a, b, c):\n    if a > b and a > c:\n        return b\n    elif b > c:\n        return b\n    return c", "label": 2},
    {"code": "total = 0\nfor i in range(1, 10):\n    total += i\nprint(total)", "label": 2},
    {"code": "def average(nums):\n    return sum(nums) / len(nums) + 1", "label": 2},
    {"code": "def celsius_to_kelvin(c):\n    return c - 273.15", "label": 2},
    {"code": "def binary_search(arr, t):\n    lo, hi = 0, len(arr)\n    while lo < hi:\n        mid = (lo + hi) // 2\n        if arr[mid] == t: return mid\n        elif arr[mid] < t: lo = mid\n        else: hi = mid - 1\n    return -1", "label": 2},
    {"code": "def fizzbuzz(n):\n    for i in range(n):\n        if i % 15 == 0: print('FizzBuzz')\n        elif i % 3 == 0: print('Fizz')\n        elif i % 5 == 0: print('Buzz')\n        else: print(i)", "label": 2},
    {"code": "def factorial(n)\n    if n = 0:\n        return 1\n    return n * factorial(n-1)", "label": 3},
    {"code": "def divide_list(nums, d):\n    results = []\n    for i in range(len(nums) + 1):\n        results.append(nums[i] / d)\n    return results\n\ndivide_list([1,2,3], 0)", "label": 3},
    {"code": "class Stack\n    def __init__(self):\n        self.items = ()\n\n    def push(self, item):\n        self.items.append(item)", "label": 3},
    {"code": "import syss\n\ndef read_file(path)\n    with open(path) as f\n        return f.read()", "label": 3},
    {"code": "def merge(a, b):\n    result = []\n    i = j = 0\n    while i < len(a) or j < len(b):\n        if a[i] < b[j]:\n            result.append(a[i])\n            i =+ 1\n        else:\n            result.append(b[j])\n            j =+ 1\n    return result", "label": 3},
]

def get_dataset():
    return RAW_DATA

def get_full_dataset(generated) :
    """Merges handcrafted + Groq-generated examples."""
    return RAW_DATA + generated

Writing /content/drive/MyDrive/code-debugger/ml_service/model/dataset.py


In [ ]:
import sys
sys.path.insert(0, "/content/drive/MyDrive/code-debugger")

from ml_service.model.dataset import RAW_DATA, LABELS, get_full_dataset
from collections import Counter

full_data=get_full_dataset(all_generated)
counts=Counter(d["label"] for d in full_data)
print(f"Handcrafted : 35")
print(f"Generated   : {len(all_generated)}")
print(f"Total       : {len(full_data)}")
print()
print("Label distribution:")
for i, name in enumerate(LABELS):
    bar = "█" * counts[i]
    print(f"  {name:<20} {counts[i]:>3}  {bar}")


Handcrafted : 35
Generated   : 212
Total       : 247

Label distribution:
  Syntax Error          65  █████████████████████████████████████████████████████████████████
  Runtime Error         56  ████████████████████████████████████████████████████████
  Logic Bug             66  ██████████████████████████████████████████████████████████████████
  Multiple Issues       60  ████████████████████████████████████████████████████████████


In [ ]:
import json
save_path="/content/drive/MyDrive/code-debugger/ml_service/model/full_dataset.json"
with open(save_path,"w") as f:
  json.dump(full_data,f,indent=2)
print(f"Saved {len(full_data)} examples to Drive")

Saved 247 examples to Drive


In [ ]:
import shutil

src = "/content/drive/MyDrive/code-debugger/ml_service/model/dataset.py"

print("dataset.py is already in the project folder")
print("full_dataset.json is already in the project folder")
print()
print("Project folder contents:")
import os
for root, dirs, files in os.walk("/content/drive/MyDrive/code-debugger"):
    level = root.replace("/content/drive/MyDrive/code-debugger", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        print(f"{indent}  {f}")

dataset.py is already in the project folder
full_dataset.json is already in the project folder

Project folder contents:
code-debugger/
  ml_service/
    model/
      README.txt
      dataset.py
      full_dataset.json
      __pycache__/
        dataset.cpython-312.pyc
  backend/
    README.txt
  frontend/
    README.txt
  notebooks/
    README.txt
